#**DEPENDENCY INSTALLATION**

In [20]:
!pip install transformers datasets sentence-transformers scikit-learn accelerate evaluate rouge_score

# **SEMANTIC_COMMENT_CLUSTERING**

In [21]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# 1. Initialize the Embedding Model
# We use 'all-distilroberta-v1' as a fast, high-performance alternative to the paper's specific RoBERTa checkpoint
embedder = SentenceTransformer('all-distilroberta-v1')

def cluster_comments(comments, distance_threshold=0.5):
    """
    Clusters a list of comments based on semantic similarity using Agglomerative Clustering.
    """
    # If there are no comments or very few, just return them as a single group to avoid errors
    if not comments:
        return {}
    if len(comments) == 1:
        return {0: comments}

    # A. Encode comments into vectors (embeddings)
    embeddings = embedder.encode(comments)

    # B. Perform Agglomerative Clustering
    # Paper Settings: Average linkage, Cosine distance, Max distance 0.5
    clustering_model = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='cosine',
        linkage='average'
    )

    clustering_model.fit(embeddings)
    cluster_assignment = clustering_model.labels_

    # C. Group comments by their cluster ID
    clustered_comments = {}
    for idx, cluster_id in enumerate(cluster_assignment):
        cluster_id = int(cluster_id) # Convert numpy int to python int for cleanliness
        if cluster_id not in clustered_comments:
            clustered_comments[cluster_id] = []
        clustered_comments[cluster_id].append(comments[idx])

    return clustered_comments

# ---------------------------------------------------------
# 2. LOAD REAL DATA FROM train.json
# ---------------------------------------------------------

# Load the JSON file
with open('/content/drive/MyDrive/data/downloadable_data/raw/train.json', 'r') as f:
    data = json.load(f)

# Let's pick a specific thread to test (e.g., the first thread in the dataset)
# You can change the index [0] to [1], [2], etc. to see different threads
selected_thread = data['threads'][9]

print(f"Processing Thread ID: {selected_thread['submission_id']}")
print(f"Subreddit: {selected_thread['subreddit']}")
print(f"Original Caption: {selected_thread['caption']}\n")

# Extract just the text body of each comment
# The JSON structure is: thread -> 'comments' -> list of dicts -> 'body'
real_comments = [c['body'] for c in selected_thread['comments']]

# Remove deleted or empty comments if necessary
real_comments = [c for c in real_comments if c not in ["[deleted]", "[removed]"] and c.strip() != ""]

print(f"Found {len(real_comments)} valid comments. Clustering now...\n")

# 3. Run the Clustering
op_clarification_text = real_comments[0]
community_comments = real_comments[1:]
clusters = cluster_comments(community_comments)

# 4. Display Results
print("-" * 30)
for cluster_id, comment_list in clusters.items():
    print(f"CLUSTER {cluster_id} ({len(comment_list)} comments):")
    for comment in comment_list:
        # Print first 100 chars to keep output readable
        preview = comment[:100] + "..." if len(comment) > 100 else comment
        print(f"  - {preview}")
    print("-" * 30)

Processing Thread ID: bgp7dn
Subreddit: malelivingspace
Original Caption: living room in my condo. any advice on wall art or a splash of color to make it feel warmer?

Found 7 valid comments. Clustering now...

------------------------------
CLUSTER 5 (1 comments):
  - Some sheer curtains under the existing curtains usually look pretty nice. Also, a couple plants woul...
------------------------------
CLUSTER 4 (1 comments):
  - Tuscan yellow (marigold, mustard, honey) accents (like sofa pillows), they pull up the blah gray and...
------------------------------
CLUSTER 3 (1 comments):
  - ID on that rug?
------------------------------
CLUSTER 2 (1 comments):
  - Since you like maps there are some colorful options out there. Instead of that severe and modern map...
------------------------------
CLUSTER 1 (1 comments):
  - You got a nice wall art but a quite small for your wide wall ! I think you need to move the lamp nea...
------------------------------
CLUSTER 0 (1 comments):
  - Too

# **SUPERVISED_FINE_TUNING_ON_MREDDITSUM**

In [22]:
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, LongT5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate
import numpy as np

# 1. Load Data from Text Files
def load_data_from_text(src_file, tgt_file):
    with open(src_file, 'r', encoding='utf-8') as f:
        sources = [line.strip() for line in f.readlines()]
    with open(tgt_file, 'r', encoding='utf-8') as f:
        targets = [line.strip() for line in f.readlines()]
    return Dataset.from_dict({"text": sources, "summary": targets})

# 2. LOAD ALL THREE DATASETS (Train: 2729, Val: 152, Test: 152)
# Ensure these paths correctly point to your Drive files
base_path = '/content/drive/MyDrive/data/downloadable_data/preprocessed/'

train_dataset = load_data_from_text(f'{base_path}train_processed_imgcap_src.txt', f'{base_path}train_processed_imgcap_tgt.txt')
val_dataset = load_data_from_text(f'{base_path}val_processed_imgcap_src.txt', f'{base_path}val_processed_imgcap_tgt.txt')
test_dataset = load_data_from_text(f'{base_path}test_processed_bestimgcap_src.txt', f'{base_path}test_processed_bestimgcap_tgt.txt')

# Combine into a proper DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

# 3. Tokenizer & Model Setup
model_checkpoint = "google/long-t5-tglobal-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = LongT5ForConditionalGeneration.from_pretrained(model_checkpoint)

# 4. Preprocessing
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 5. Metrics
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# 6. Updated Training Arguments (Matching paper's 50 epochs and 3e-5 learning rate)
args = Seq2SeqTrainingArguments(
    output_dir="./t5-mredditsum-final-splits",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # Paper's recommended rate [cite: 719]
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=20,         # Paper trained for 50 epochs
    predict_with_generate=True,
    bf16=True,
    load_best_model_at_end=True, # Critical: Uses validation set to pick best model
    metric_for_best_model="rouge1",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Training monitors validation performance
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

# 7. Train
trainer.train()

# 8. FINAL EVALUATION ON THE TEST SET
print("\n--- Final Evaluation on Unseen Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test ROUGE Scores: {test_results}")

# 9. Save
trainer.save_model("./final_mredditsum_long-t5-tglobal-base_with_img_caption_20_epoch")

Map:   0%|          | 0/2729 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

/tmp/ipython-input-1553247717.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.438100,1.758547,21.349900,11.966400,19.073400,19.024900
2,2.119800,1.718616,21.171700,11.962000,18.932500,18.897600
3,1.952000,1.683279,21.438900,12.204900,19.299100,19.257000
4,1.906500,1.666553,21.658900,12.427700,19.479000,19.462000
5,1.882800,1.669192,21.132700,11.885300,19.082900,19.042400
6,1.818300,1.658347,21.567400,12.173600,19.268400,19.246700
7,1.808200,1.650863,21.313100,11.976800,19.063200,19.038500
8,1.779100,1.647501,21.180900,11.787100,18.915800,18.871500
9,1.729000,1.640909,21.005400,11.824800,18.690700,18.654000
10,1.730800,1.644457,21.263800,12.073200,18.900100,18.895500



--- Final Evaluation on Unseen Test Set ---


Test ROUGE Scores: {'eval_loss': 1.7288838624954224, 'eval_rouge1': 20.8743, 'eval_rouge2': 11.3823, 'eval_rougeL': 18.5187, 'eval_rougeLsum': 18.5146, 'eval_runtime': 26.8438, 'eval_samples_per_second': 5.662, 'eval_steps_per_second': 1.416, 'epoch': 20.0}


# **SINGLE_PASS_BASELINE_INFERENCE**

In [27]:
from transformers import pipeline

# Load your fine-tuned model
summarizer = pipeline("summarization", model="/content/final_mredditsum_long-t5-tglobal-base_with_img_caption_20_epoch", tokenizer=tokenizer, device=0)

# Example input (You can paste a raw line from your src.txt here)
input_text = "Original Post: What color should I paint my walls? Image: A living room with beige furniture. OP: I feel like it's too boring. User 1: Try sage green! User 2: I agree, green would look great."

# T5 prompt prefix
input_text = "summarize: " + input_text

summary = summarizer(input_text, max_length=128, min_length=30, do_sample=False)
print("Generated Summary:", summary[0]['summary_text'])

Device set to use cuda:0
Your max_length is set to 128, but your input_length is only 56. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Summary: OP asked what color they should paint the walls in their living room with beige furniture. One commenter suggested sage green. Another commenter suggested sage green.


# **CMS_PIPELINE_EXECUTION_(SUMMARIZATION_&_SYNTHESIS)**

In [24]:
def run_cms_pipeline(thread_json_object, model, tokenizer):
    """
    Uses the raw JSON object to ensure we cluster real COMMENTS, not words.
    """
    # 1. Extract real comments from JSON
    real_comments = [c['body'] for c in thread_json_object['comments']]
    real_comments = [c for c in real_comments if c not in ["[deleted]", "[removed]"] and c.strip() != ""]

    if len(real_comments) < 2:
        return "Not enough comments to summarize."

    # First comment is usually OP clarification
    op_clarification = real_comments[0]
    community_comments = real_comments[1:]
    op_caption = thread_json_object.get('caption', 'No image available')

    # 2. Stage 1: Clustering (Using your new dynamic function)
    clusters = cluster_comments(community_comments)

    # 3. Stage 2: Summarize Clusters
    op_sum, cluster_sums = stage_2_summarize_clusters_fixed(
        clusters, op_caption, op_clarification, model, tokenizer
    )

    # 4. Stage 3: Synthesis
    final_summary = stage_3_synthesis(op_sum, cluster_sums, model, tokenizer)

    return final_summary

In [26]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")
cms_predictions = []
ground_truth_references = []

# Assuming you have loaded your JSON: data = json.load(f)
# We need to find where the 'test' threads start.
# Usually, in MREDDITSUM, the last 152 threads are the test set.
test_threads = data['threads'][-152:]

print(f"Starting CMS Evaluation on {len(test_threads)} test threads...")

for i in tqdm(range(len(test_threads))):
    thread_obj = test_threads[i]

    try:
        # 1. Generate the prediction
        pred = run_cms_pipeline(thread_obj, model, tokenizer)

        # 2. Get the ground truth using the correct key we just found
        truth = thread_obj.get('edited_sum')

        if pred and truth:
            cms_predictions.append(pred)
            ground_truth_references.append(truth)

    except Exception as e:
        print(f"Error on thread {i}: {e}")
        continue

# Calculate the scores
results = rouge.compute(predictions=cms_predictions, references=ground_truth_references)
print("\nCORRECTED CMS ROUGE SCORES:", results)

Starting CMS Evaluation on 152 test threads...


  0%|          | 0/152 [00:00<?, ?it/s]

Stage 2: Summarizing OP (Caption + Text) and 9 comment clusters...
Stage 3: Synthesizing final summary...


  1%|          | 1/152 [00:12<32:13, 12.80s/it]

Stage 2: Summarizing OP (Caption + Text) and 11 comment clusters...
Stage 3: Synthesizing final summary...


...

Stage 2: Summarizing OP (Caption + Text) and 5 comment clusters...
Stage 3: Synthesizing final summary...


 99%|█████████▉| 151/152 [28:30<00:10, 10.80s/it]

Stage 2: Summarizing OP (Caption + Text) and 19 comment clusters...
Stage 3: Synthesizing final summary...


100%|██████████| 152/152 [28:55<00:00, 11.42s/it]



CORRECTED CMS ROUGE SCORES: {'rouge1': np.float64(0.42343025057886574), 'rouge2': np.float64(0.1747656500248405), 'rougeL': np.float64(0.2850343689150474), 'rougeLsum': np.float64(0.32972621206449504)}


**Getting rogue scores for T5-Base(with image) and bart_base(with and without image)**


In [29]:
def evaluate_saved_model(model_path, model_type="t5"):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    print(f"\n--- Loading {model_type.upper()} from {model_path} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    # Automatically loads BART or T5 weights correctly
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")

    cms_preds = []
    truths = []

    # We use a subset (e.g., 50 threads) if you are very short on time,
    # or the full 152 for paper-ready results.
    test_threads = data['threads'][-152:]

    for i in tqdm(range(len(test_threads))):
        thread_obj = test_threads[i]
        try:
            pred = run_cms_pipeline(thread_obj, model, tokenizer)
            truth = thread_obj.get('edited_sum')
            if pred and truth:
                cms_preds.append(pred)
                truths.append(truth)
        except Exception as e:
            continue

    results = rouge.compute(predictions=cms_preds, references=truths)
    # Multiply by 100 for readability (0.43 -> 43.0)
    final_scores = {k: round(v * 100, 2) for k, v in results.items()}
    return final_scores

In [30]:
import pandas as pd

# Define your model paths and labels
model_configs = [
    {"path": "/content/final_mredditsum_model_bart_base_with_img_caption_20_epoch", "label": "BART (With Img)"},
    {"path": "/content/final_mredditsum_model_bart_base_without_img_caption_20_epoch", "label": "BART (No Img)"},
    {"path": "/content/final_mredditsum_model_with_img_caption_20_epoch", "label": "T5-Base (With Img)"},
]

all_results = []

for config in model_configs:
    print(f"\n" + "="*50)
    print(f"EVALUATING: {config['label']}")
    print("="*50)

    # Run the evaluation
    scores = evaluate_saved_model(config['path'], model_type="auto")

    # Add label for the table
    scores['Model Name'] = config['label']
    all_results.append(scores)

# 2. Generate Final Comparison Table
df_results = pd.DataFrame(all_results)
# Reorder columns for readability
df_results = df_results[['Model Name', 'rouge1', 'rouge2', 'rougeL', 'rougeLsum']]

print("\n\n--- FINAL SYSTEM COMPARISON ---")
print(df_results)


EVALUATING: BART (With Img)

--- Loading AUTO from /content/final_mredditsum_model_bart_base_with_img_caption_20_epoch ---


  0%|          | 0/152 [00:00<?, ?it/s]

Stage 2: Summarizing OP (Caption + Text) and 9 comment clusters...
Stage 3: Synthesizing final summary...


  1%|          | 1/152 [00:05<12:55,  5.13s/it]

Stage 2: Summarizing OP (Caption + Text) and 11 comment clusters...
Stage 3: Synthesizing final summary...


...

Stage 2: Summarizing OP (Caption + Text) and 5 comment clusters...
Stage 3: Synthesizing final summary...


 99%|█████████▉| 151/152 [22:24<00:09,  9.13s/it]

Stage 2: Summarizing OP (Caption + Text) and 19 comment clusters...
Stage 3: Synthesizing final summary...


100%|██████████| 152/152 [22:43<00:00,  8.97s/it]




--- FINAL SYSTEM COMPARISON ---
           Model Name  rouge1  rouge2  rougeL  rougeLsum
0     BART (With Img)   45.17   19.19   29.94      34.89
1       BART (No Img)   46.78   19.30   29.50      34.43
2  T5-Base (With Img)   41.40   14.48   24.20      30.22
